# 🔬 Notebook 05: VisReg-3D Masking Diagnostics (D1–D3)
### Validates brain-aware masking + tissue-only regularization on real BraTS data

This notebook runs `scripts/diagnose_visreg_3d.py`:
- **D1 (no checkpoint):** context/target air fractions on real volumes — did uniform sampling waste the signal?
- **D2/D3 (needs `visreg_jepa_3d_best.pt`):** air-vs-tissue prediction loss split + VisReg scale/shape on all vs tissue-only tokens.

> **Estimated runtime:** ~10–15 min on T4 (~5 setup, ~3 D1 on 200 vols, ~5 D2/D3 on 50 vols).


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


## 2. Experiment Configuration
Set `CHECKPOINT_PATH` to your pre-trained `visreg_jepa_3d_best.pt` (e.g. `/kaggle/input/<your-ckpt-dataset>/visreg_jepa_3d_best.pt`). Leave `""` to run D1 only.


In [ ]:
SEED = 42
NUM_WORKERS = 4
NUM_VOLUMES = 200  # D1 volumes (D2/D3 cap at 50 internally)
CHECKPOINT_PATH = ""  # e.g. "/kaggle/input/visreg-ckpt/visreg_jepa_3d_best.pt"

# Belt-and-suspenders: pin the mounted full-pool dataset so every script
# resolving the default name finds it even without an explicit data_dir.
import os as _os
from pathlib import Path as _Path
_kaggle_full = _Path("/kaggle/input/brats-3d-full/brats_gli_3d_full")
if _kaggle_full.is_dir() and (_kaggle_full / "metadata.csv").exists():
    _os.environ["BRATS3D_DATA_DIR"] = str(_kaggle_full)
    print(f"Dataset env override: BRATS3D_DATA_DIR={_kaggle_full}")


## 3. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"MONAI Version:    v{monai.__version__}")
print(f"NiBabel Version:  v{nib.__version__}")


## 4. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/kaggle/working/tumor_segmentation_3d")

if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

if not (work_dir / "src" / "brats_jepa_3d").exists():
    print(f"Cloning codebase from: {REPO_URL} ...")
    !git clone {REPO_URL} {work_dir}

src_dir = work_dir / "src"
if not (src_dir / "brats_jepa_3d").exists():
    raise RuntimeError("Clone failed! Toggle Internet ON in the Kaggle notebook settings.")

os.chdir(str(work_dir))
%cd {work_dir}

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

!pip install -q -e .

print(f"Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"


## 5. Dataset Discovery & Health Checks
Attach the **`brats-3d-full`** dataset via **+ Add Input**. Expects `brats_gli_3d_full/` with `metadata.csv` (1,621 scans; test $N=242$).


In [ ]:
import pandas as pd
import torch

from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

print("=== DATASET DISCOVERY ===")
data_dir = get_dataset_dir("brats_gli_3d_full")
meta_path = get_metadata_path("brats_gli_3d_full")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

if not meta_path.exists():
    raise RuntimeError("Attach 'brats-3d-full' via '+ Add Input' in Kaggle!")

df = pd.read_csv(meta_path)
print(f"Loaded Metadata: {len(df)} records")
print(df["split"].value_counts().to_string())


## 6. D1: Mask Air-Fraction on Real Volumes (no checkpoint)
Target: context air ≈ 55% (harmless post-fix), target air < 30% (was 35.8% uniform on proxy).


In [ ]:
!python scripts/diagnose_visreg_3d.py --num_volumes {NUM_VOLUMES} --seed {SEED}


## 7. Checkpoint Resolution (for D2/D3)
Point `CHECKPOINT_PATH` (§2) at `visreg_jepa_3d_best.pt`, or upload the file to `/kaggle/working` and re-run §2. Auto-detects `*visreg*best.pt` under `/kaggle/working` and `/kaggle/input` as fallback.


In [ ]:
from pathlib import Path

ckpt = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else None
if ckpt is None or not ckpt.exists():
    cands = list(Path("/kaggle/working").glob("**/*visreg*best.pt")) + \
            list(Path("/kaggle/input").glob("**/*visreg*best.pt"))
    ckpt = Path(cands[-1]) if cands else None
print(f"Checkpoint for D2/D3: {ckpt if ckpt and ckpt.exists() else 'NONE — D1 only'}")


## 8. D2/D3: Prediction Split + Regularization Probe (needs checkpoint)
Target: tissue prediction loss dominates; scale/shape on tissue-only ≪ all-token (≈19–23× gap pre-fix on proxy).


In [ ]:
if ckpt is not None and ckpt.exists():
    !python scripts/diagnose_visreg_3d.py --num_volumes {NUM_VOLUMES} --seed {SEED} --checkpoint {ckpt}
else:
    print("Skipping D2/D3: no checkpoint. D1 above already answers the go/no-go.")


## 9. Verdict: Go / No-Go for Full Pretraining
- **GO** if target air < 30% with disjointness holding and (when run) tissue loss dominating + tissue-only scale/shape ≪ all-token.
- **NO-GO** (rethink masking) if target air stays ≈ uniform levels or all-air target blocks appear.
Next on GO: 100-epoch pretrain → 30-epoch finetune off/on `--deep_supervision` (notebook 01), expecting full-data Dice above the 79.71% baseline toward 86–87%.
